# 01 - Data preparation

Filters Spider (+ optional BIRD) dev sets, serializes schemas, and writes ChatML JSONL records using the `zeroerr` package.

In [ ]:
# Install the package in editable mode (training extras optional)
# pip install -e ".[train]"

In [ ]:
from pathlib import Path
import json

In [ ]:
# 1. Download Spider reformat from HF hub (mirrors the official tar).
from datasets import load_dataset

ds = load_dataset("xlangai/spider", split="train")
rows = [{"db_id": r["db_id"], "question": r["question"], "query": r["query"]} for r in ds]

with open("data/raw/spider_train.jsonl", "w") as fh:
    for row in rows:
        fh.write(json.dumps(row) + "\n")

In [ ]:
# 2. Optional: merge in a curated slice of BIRD-SQL dev (queries with FK metadata).# 3. Run the filtering + formatting CLI shipped with the repo.
!python -m zeroerr.data.prep \
    -i data/raw/spider_train.jsonl \
    -o data/chatml/train.jsonl \
    --per-bucket 2000 --with-repairs

In [ ]:
# Inspect a batch example.
from zeroerr.data.chatml import ChatExample, render_chatml
from zeroerr.data.schemas import render_ddl

example = ChatExample(schema_text="CREATE TABLE t (department_id, name TEXT)",
                      question="List departments",
                      answer="SELECT name FROM t")
print(render_chatml(example.to_messages()))

Expected output artifact:
`data/chatml/train.jsonl` with `text` (rendered ChatML) and `plain` (role/content) fields.